# Diffusion Models on Galaxy Images

A minimal diffusion model trained on 32×32 galaxy images. Demonstrates:
1. **Forward process** — noising images at different timesteps
2. **Training** — predict the noise, take MSE
3. **Generation** — start from pure noise, denoise step by step

In [ ]:
from functools import partial

import flax.linen as nn
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
import optax

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["font.size"] = 12

## Load data

In [ ]:
data = np.load("../assignments/lab_04/galaxy_data.npz")
X_train = jnp.array(data["X_train"].astype(np.float32) / 255.0)
print(f"Training data: {X_train.shape} — values in [{float(X_train.min()):.1f}, {float(X_train.max()):.1f}]")

## Noise schedule

Linear schedule for $\beta_t$, precompute $\bar\alpha_t = \prod_{s=1}^t (1 - \beta_s)$.

In [ ]:
T = 200

beta = jnp.linspace(1e-4, 0.02, T)
alpha = 1.0 - beta
alpha_bar = jnp.cumprod(alpha)

plt.plot(alpha_bar)
plt.xlabel("Timestep t")
plt.ylabel(r"$\bar{\alpha}_t$")
plt.title("Signal retained over time")
plt.grid(True, alpha=0.3)
plt.show()

## Forward process: the diffusion kernel

$$z_t = \sqrt{\bar\alpha_t}\, x + \sqrt{1 - \bar\alpha_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

No need to run the chain — jump directly to any $t$.

In [ ]:
def q_sample(x, t, key):
    """Sample z_t from the diffusion kernel."""
    eps = jr.normal(key, x.shape)
    a = jnp.sqrt(alpha_bar[t])[:, None, None, None]
    b = jnp.sqrt(1.0 - alpha_bar[t])[:, None, None, None]
    return a * x + b * eps, eps


# Show one galaxy at increasing noise levels
x0 = X_train[42:43]  # (1, 32, 32, 1)
timesteps = [0, 20, 50, 100, 150, 199]

fig, axes = plt.subplots(1, len(timesteps), figsize=(14, 2.5))
for i, t_val in enumerate(timesteps):
    t = jnp.array([t_val])
    z_t, _ = q_sample(x0, t, jr.PRNGKey(0))
    axes[i].imshow(np.array(z_t[0, :, :, 0]), cmap="gray")
    axes[i].set_title(f"t = {t_val}")
    axes[i].axis("off")
plt.suptitle("Forward process: clean \u2192 noise", fontsize=14)
plt.tight_layout()
plt.show()

## Denoiser network

A small U-Net that takes $(z_t, t)$ and predicts the noise $\hat\epsilon$. Time is injected via sinusoidal embeddings.

In [ ]:
def sinusoidal_embedding(t, dim=64):
    """Sinusoidal positional embedding for timestep t."""
    half = dim // 2
    freqs = jnp.exp(-jnp.log(10000.0) * jnp.arange(half) / half)
    args = t[:, None].astype(jnp.float32) * freqs[None, :]
    return jnp.concatenate([jnp.sin(args), jnp.cos(args)], axis=-1)


class SimpleUNet(nn.Module):
    """Minimal U-Net for 32x32 images."""

    @nn.compact
    def __call__(self, z_t, t):
        # Time embedding: sinusoidal -> dense
        te = sinusoidal_embedding(t, 64)
        te = nn.relu(nn.Dense(64)(te))

        # --- Down ---
        h1 = nn.relu(nn.GroupNorm(8)(nn.Conv(32, (3, 3))(z_t)))  # (B, 32, 32, 32)
        h1 = h1 + nn.Dense(32)(te)[:, None, None, :]

        h2 = nn.relu(nn.GroupNorm(8)(nn.Conv(64, (3, 3), strides=(2, 2))(h1)))  # (B, 16, 16, 64)
        h2 = h2 + nn.Dense(64)(te)[:, None, None, :]

        h3 = nn.relu(nn.GroupNorm(8)(nn.Conv(64, (3, 3), strides=(2, 2))(h2)))  # (B, 8, 8, 64)
        h3 = h3 + nn.Dense(64)(te)[:, None, None, :]

        # --- Up ---
        h = jax.image.resize(h3, (*h2.shape[:3], h3.shape[3]), method="nearest")
        h = jnp.concatenate([h, h2], axis=-1)
        h = nn.relu(nn.GroupNorm(8)(nn.Conv(64, (3, 3))(h)))

        h = jax.image.resize(h, (*h1.shape[:3], h.shape[3]), method="nearest")
        h = jnp.concatenate([h, h1], axis=-1)
        h = nn.relu(nn.GroupNorm(8)(nn.Conv(32, (3, 3))(h)))

        return nn.Conv(1, (3, 3))(h)  # predict noise, same shape as input


# Check shapes
model = SimpleUNet()
key = jr.PRNGKey(0)
dummy_z = jnp.ones((2, 32, 32, 1))
dummy_t = jnp.array([10, 50])
params = model.init(key, dummy_z, dummy_t)
out = model.apply(params, dummy_z, dummy_t)
n_params = sum(p.size for p in jax.tree.leaves(params))
print(f"Output shape: {out.shape}")
print(f"Parameters: {n_params:,}")
assert out.shape == (2, 32, 32, 1), f"Expected (2, 32, 32, 1), got {out.shape}"

## Training

The loss is simple:
1. Sample $x$, sample $t$, sample $\epsilon$
2. Compute $z_t = \sqrt{\bar\alpha_t}\, x + \sqrt{1 - \bar\alpha_t}\, \epsilon$
3. Loss: $\|\epsilon - \hat\epsilon_\theta(z_t, t)\|^2$

In [ ]:
optimizer = optax.adam(3e-4)


def loss_fn(params, x_batch, key):
    """Diffusion training loss: MSE on noise prediction."""
    k1, k2 = jr.split(key)
    batch_size = x_batch.shape[0]

    # Sample random timesteps and noise
    t = jr.randint(k1, (batch_size,), 0, T)
    z_t, eps = q_sample(x_batch, t, k2)

    # Predict noise
    eps_hat = model.apply(params, z_t, t)

    return jnp.mean((eps - eps_hat) ** 2)


@jax.jit
def train_step(params, opt_state, x_batch, key):
    loss, grads = jax.value_and_grad(loss_fn)(params, x_batch, key)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

In [ ]:
# Initialize
key = jr.PRNGKey(42)
key, init_key = jr.split(key)
params = model.init(init_key, jnp.ones((1, 32, 32, 1)), jnp.array([0]))
opt_state = optimizer.init(params)

# Training
n_epochs = 15
batch_size = 256
n_train = len(X_train)
rng = np.random.default_rng(42)
losses = []

for epoch in range(n_epochs):
    idx = rng.permutation(n_train)
    epoch_losses = []

    for start in range(0, n_train, batch_size):
        key, step_key = jr.split(key)
        batch = X_train[idx[start : start + batch_size]]
        params, opt_state, loss = train_step(params, opt_state, batch, step_key)
        epoch_losses.append(float(loss))

    avg_loss = np.mean(epoch_losses)
    losses.append(avg_loss)
    print(f"Epoch {epoch + 1:2d}/{n_epochs} \u2014 loss: {avg_loss:.4f}")

plt.plot(range(1, len(losses) + 1), losses, "b-o", markersize=5)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training loss")
plt.grid(True, alpha=0.3)
plt.show()

## Generation

Start from pure noise $z_T \sim \mathcal{N}(0, I)$, denoise step by step:
1. Predict noise $\hat\epsilon = \hat\epsilon_\theta(z_t, t)$
2. Estimate clean image $\hat z_0$
3. Sample $z_{t-1}$ from the reverse conditional

In [ ]:
@partial(jax.jit, static_argnums=(2,))
def generate(params, key, n_samples=16):
    """Generate samples via the full reverse chain."""
    z = jr.normal(key, (n_samples, 32, 32, 1))

    def step(z, t_idx):
        # t_idx counts down from T-1 to 0
        t = jnp.full((n_samples,), t_idx)
        eps_hat = model.apply(params, z, t)

        # Estimate clean image
        ab = alpha_bar[t_idx]
        z0_hat = (z - jnp.sqrt(1 - ab) * eps_hat) / jnp.sqrt(ab)

        # Reverse conditional mean
        ab_prev = jnp.where(t_idx > 0, alpha_bar[t_idx - 1], 1.0)
        b = beta[t_idx]
        mu = (jnp.sqrt(ab_prev) * b / (1 - ab)) * z0_hat + (jnp.sqrt(1 - b) * (1 - ab_prev) / (1 - ab)) * z

        # Variance and noise injection (skip at t=0)
        var = b * (1 - ab_prev) / (1 - ab)
        key = jr.fold_in(jr.PRNGKey(0), t_idx)
        noise = jr.normal(key, z.shape)
        z = mu + jnp.where(t_idx > 0, jnp.sqrt(var) * noise, 0.0)
        return z, z

    # Run reverse chain from T-1 down to 0
    z_final, z_trajectory = jax.lax.scan(step, z, jnp.arange(T - 1, -1, -1))
    return z_final, z_trajectory


key, gen_key = jr.split(key)
samples, trajectory = generate(params, gen_key, n_samples=16)

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(np.clip(np.array(samples[i, :, :, 0]), 0, 1), cmap="gray")
    ax.axis("off")
plt.suptitle("Generated galaxies (from pure noise)", fontsize=14)
plt.tight_layout()
plt.show()

## Denoising trajectory

Watch a single sample evolve from noise to galaxy:

In [ ]:
# trajectory shape: (T, n_samples, 32, 32, 1) — from t=T-1 down to t=0
show_steps = [0, 20, 50, 100, 150, 180, 195, 199]  # indices into the trajectory (0 = noisiest)

fig, axes = plt.subplots(1, len(show_steps), figsize=(14, 2.5))
for i, s in enumerate(show_steps):
    t_val = T - 1 - s  # convert trajectory index to actual timestep
    axes[i].imshow(np.clip(np.array(trajectory[s, 0, :, :, 0]), 0, 1), cmap="gray")
    axes[i].set_title(f"t = {t_val}")
    axes[i].axis("off")
plt.suptitle("Reverse process: noise \u2192 galaxy", fontsize=14)
plt.tight_layout()
plt.show()

## Real vs generated

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i in range(8):
    axes[0, i].imshow(np.array(X_train[i, :, :, 0]), cmap="gray")
    axes[1, i].imshow(np.clip(np.array(samples[i, :, :, 0]), 0, 1), cmap="gray")
for ax in axes.flat:
    ax.axis("off")
axes[0, 0].set_ylabel("Real", fontsize=12, rotation=90, labelpad=10)
axes[1, 0].set_ylabel("Generated", fontsize=12, rotation=90, labelpad=10)
plt.suptitle("Real vs generated galaxies", fontsize=14)
plt.tight_layout()
plt.show()